# Export HYDE 3.4 Land Cover for Lithuania

Exports:
- **PNG rasters** (dashboard style)
- **GeoTIFF rasters** (`rasters/hyde/geotiff/`) for web display – zoomable
- **CSV** per-class area counts

Run all cells. Use a kernel where rasterio works for GeoTIFF export.

In [ ]:
from pathlib import Path
import json
from shapely.geometry import shape, Point
from shapely.ops import unary_union
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

BASE = Path(r"C:\Users\matas\Desktop\LEI\Data")
LT_GEOJSON = BASE / "lt_boundary_admin.json"
HYDE_NC_DIR = BASE / "Hyde" / "34" / "NetCDF"
OUT_RASTERS = BASE / "rasters" / "hyde"
OUT_GEOTIFF = OUT_RASTERS / "geotiff"
OUT_CSV = BASE / "outputs" / "hyde_lithuania_timeseries.csv"
OUT_RASTERS.mkdir(parents=True, exist_ok=True)
OUT_GEOTIFF.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with open(LT_GEOJSON, "r", encoding="utf-8") as f:
    lt_data = json.load(f)
geoms = [shape(feat["geometry"]) for feat in lt_data["features"]]
lt_geom = unary_union(geoms)

def polygon_mask(lat_vals, lon_vals, geom):
    h, w = len(lat_vals), len(lon_vals)
    mask = np.zeros((h, w), dtype=bool)
    for i, lat in enumerate(lat_vals):
        for j, lon in enumerate(lon_vals):
            mask[i, j] = geom.contains(Point(lon, lat))
    return mask

LAT_MIN, LAT_MAX = 53.5, 56.6
LON_MIN, LON_MAX = 20.5, 26.7

cropland_path = HYDE_NC_DIR / "cropland.nc"
urban_path = HYDE_NC_DIR / "urban_area.nc"
pasture_path = HYDE_NC_DIR / "pasture.nc"
ds_crop = xr.open_dataset(cropland_path, chunks="auto")
ds_urban = xr.open_dataset(urban_path, chunks="auto")
crop_da = ds_crop[list(ds_crop.data_vars)[0]]
urban_da = ds_urban[list(ds_urban.data_vars)[0]]
lat_name = "lat" if "lat" in crop_da.coords else "latitude"
lon_name = "lon" if "lon" in crop_da.coords else "longitude"
lon_in_360 = float(np.max(crop_da[lon_name].values)) > 180
lon_lo, lon_hi = (LON_MIN + 360, LON_MAX + 360) if lon_in_360 else (LON_MIN, LON_MAX)

def try_crop(da):
    for lat_sl in [slice(LAT_MIN, LAT_MAX), slice(LAT_MAX, LAT_MIN)]:
        out = da.sel(**{lat_name: lat_sl, lon_name: slice(lon_lo, lon_hi)})
        if out.sizes.get(lat_name, 0) > 0:
            return out
    return None

crop_lt = try_crop(crop_da)
urban_lt = try_crop(urban_da)
crop_lt, urban_lt = xr.align(crop_lt, urban_lt, join="inner")
if pasture_path.exists():
    ds_past = xr.open_dataset(pasture_path, chunks="auto")
    past_lt = try_crop(ds_past[list(ds_past.data_vars)[0]])
    if past_lt is not None:
        crop_lt, urban_lt, past_lt = xr.align(crop_lt, urban_lt, past_lt, join="inner")
        agriculture = crop_lt + past_lt
    else:
        agriculture = crop_lt
else:
    agriculture = crop_lt

lat_vals = crop_lt[lat_name].values
lon_vals = crop_lt[lon_name].values
if len(lat_vals) > 1 and lat_vals[0] < lat_vals[-1]:
    crop_lt = crop_lt.isel({lat_name: slice(None, None, -1)})
    urban_lt = urban_lt.isel({lat_name: slice(None, None, -1)})
    agriculture = agriculture.isel({lat_name: slice(None, None, -1)})
    lat_vals = crop_lt[lat_name].values

mask = polygon_mask(lat_vals, lon_vals, lt_geom)
other = 1.0 - urban_lt - agriculture
other = other.where(other > 0, 0)

class_names = {1: "Water", 2: "Wetland", 3: "Urban", 4: "Agriculture", 5: "Forest"}
colors = ["#4DA6FF", "#7B68EE", "#FF4D4D", "#FFD24D", "#228B22"]
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = matplotlib.colors.Normalize(vmin=1, vmax=5)

time_vals = crop_lt["time"].values
all_years = np.array([t.year if hasattr(t, "year") else int(t) for t in time_vals], dtype=int)
years = np.unique(all_years)
years = years[(years >= 1900) & (years <= 2020)]
print("Exporting HYDE years:", list(years[:10]), "...", list(years[-5:]))

Exporting HYDE years: [np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1951), np.int64(1952), np.int64(1953), np.int64(1954)] ... [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]


In [2]:
records = []
for year in years:
    idx_arr = np.where(all_years == year)[0]
    if len(idx_arr) == 0:
        continue
    idx = int(idx_arr[0])
    u = urban_lt.isel(time=idx).values
    a = agriculture.isel(time=idx).values
    o = other.isel(time=idx).values
    dom = np.argmax(np.stack([u, a, o], axis=-1), axis=-1)
    arr = np.full(dom.shape, np.nan, dtype="float32")
    arr[dom == 0] = 3
    arr[dom == 1] = 4
    arr[dom == 2] = 5
    arr_masked = np.where(mask, arr, np.nan)

    flat = arr_masked[np.isfinite(arr_masked)].astype(int)
    if flat.size:
        uniq, cnts = np.unique(flat, return_counts=True)
        for cls_id, cnt in zip(uniq, cnts):
            records.append((int(year), int(cls_id), class_names[int(cls_id)], int(cnt)))

    rgba = cmap(norm(arr_masked))
    plt.imsave(OUT_RASTERS / f"hyde_{int(year)}.png", rgba)

    h, w = arr_masked.shape
    west, east = float(np.min(lon_vals)), float(np.max(lon_vals))
    south, north = float(np.min(lat_vals)), float(np.max(lat_vals))
    arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)
    transform = from_bounds(west, south, east, north, w, h)
    out_tif = OUT_GEOTIFF / f"hyde_{int(year)}.tif"
    with rasterio.open(out_tif, "w", driver="GTiff", height=h, width=w, count=1,
                      dtype=arr_uint8.dtype, crs="EPSG:4326", transform=transform, nodata=0) as dst:
        dst.write(arr_uint8, 1)

    if int(year) % 20 == 0:
        print(f"  Saved hyde_{year}.png, hyde_{year}.tif")

print("Done.")

C:\Users\matas\AppData\Local\Temp\ipykernel_1208\3899183918.py:29: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)


  Saved hyde_1900.png, hyde_1900.tif
  Saved hyde_1920.png, hyde_1920.tif
  Saved hyde_1940.png, hyde_1940.tif
  Saved hyde_1960.png, hyde_1960.tif
  Saved hyde_1980.png, hyde_1980.tif
  Saved hyde_2000.png, hyde_2000.tif
  Saved hyde_2020.png, hyde_2020.tif
Done.


In [3]:
df = pd.DataFrame(records, columns=["year", "class_id", "class_name", "count"])
df.to_csv(OUT_CSV, index=False)
print("Saved CSV:", OUT_CSV)
df.head(10)

Saved CSV: C:\Users\matas\Desktop\LEI\Data\outputs\hyde_lithuania_timeseries.csv


,year,class_id,class_name,count
0,1900,3,Urban,1
1,1900,4,Agriculture,1231
2,1900,5,Forest,78
3,1910,3,Urban,1
4,1910,4,Agriculture,1231
5,1910,5,Forest,78
6,1920,3,Urban,1
7,1920,4,Agriculture,1230
8,1920,5,Forest,79
9,1930,3,Urban,2
